In [ ]:
Использовал локальную модель, какие то проблемы были с hugging face, 

In [1]:
import os
import gc
import re
import json
import random
import urllib.request
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# =============================================================================
# SETUP and MODEL LOADING
# =============================================================================
torch.cuda.empty_cache()
gc.collect()

# Dynamically select the best available hardware
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
print(f"Using device: {device}")

model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ" 
print(f"Loading tokenizer and model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

# Let Hugging Face handle the optimal device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",             
    torch_dtype=torch.float16, 
    low_cpu_mem_usage=True,
    attn_implementation="sdpa" 
)

def generate_local(prompt: str, max_new_tokens: int = 100, temp: float = 0.7, do_sample: bool = True) -> str:
    """Helper function to generate text from the model."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id
    }
    
    if do_sample:
        gen_kwargs["temperature"] = temp
        
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        
    # Decode only the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()


# =============================================================================
# PART 1: PROMPT ENGINEERING (Tasks 1-4)
# =============================================================================
def run_part_1():
    print("\n" + "=" * 80)
    print("PART 1: PROMPT ENGINEERING")
    print("=" * 80)

    print("\n--- TASK 1: CHARACTERS ---")
    prompt_a = "Gordon Ramsay: This risotto is absolutely dreadful!\nSpongeBob SquarePants: "
    print("[Setup A - Names Only]")
    print(generate_local(prompt_a, temp=0.8, max_new_tokens=60))

    prompt_b = (
        "Context: Gordon Ramsay is visiting the Krusty Krab to review their food. "
        "SpongeBob is trying to impress him but is very nervous.\n"
        "Gordon Ramsay: This Krabby Patty is absolutely dreadful! It's soaking in grease!\n"
        "SpongeBob SquarePants: "
    )
    print("\n[Setup B - With Context]")
    print(generate_local(prompt_b, temp=0.8, max_new_tokens=60))

    print("\n--- TASK 2: TRANSLATION ---")
    translation_prompt = (
        "Translate the following English poem excerpt into French.\n\n"
        "English:\nOnce upon a midnight dreary, while I pondered, weak and weary,\n"
        "Over many a quaint and curious volume of forgotten lore—\n\nFrench:\n"
    )
    print(generate_local(translation_prompt, temp=0.3, max_new_tokens=60))

    print("\n--- TASK 3: PRONOUNS ---")
    pronoun_prompt = (
        "Rewrite the following sentences to change the gender pronouns of the main actor from male to female.\n\n"
        "Input: The doctor took off his mask and wiped his forehead.\n"
        "Output: The doctor took off her mask and wiped her forehead.\n\n"
        "Input: He walked to his car and unlocked it with his keys.\n"
        "Output: She walked to her car and unlocked it with her keys.\n\n"
        "Input: The pilot adjusted his headset before he started the engines.\n"
        "Output:\n"
    )
    print(generate_local(pronoun_prompt, temp=0.1, max_new_tokens=50))

    print("\n--- TASK 4: IMPERIAL TO METRIC ---")
    metric_prompt = (
        "Convert all imperial units in the sentences to metric units (miles to kilometers, mph to km/h). "
        "Note: 1 mile is approximately 1.6 km.\n\n"
        "Input: The speed limit is 60 mph.\n"
        "Output: The speed limit is 96 km/h.\n\n"
        "Input: I ran 5 miles yesterday.\n"
        "Output: I ran 8 kilometers yesterday.\n\n"
        "Input: The nearest gas station is 10 miles away, and you have to drive at 45 mph.\n"
        "Output:\n"
    )
    print(generate_local(metric_prompt, temp=0.1, max_new_tokens=50))


# =============================================================================
# PART 2: NUCLEUS SAMPLING (Task 5)
# =============================================================================
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
        
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    sorted_indices_to_remove = cumulative_probs > prob
    sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
    sorted_indices_to_remove[0] = False 
    
    indices_to_keep = sorted_indices[~sorted_indices_to_remove]
    valid_probs = sorted_probs[~sorted_indices_to_remove]
    valid_probs = valid_probs / torch.sum(valid_probs)
    
    next_token_idx_in_valid = torch.multinomial(valid_probs, num_samples=1)
    next_token_id = indices_to_keep[next_token_idx_in_valid].item()
    
    sampled_token = tokenizer.decode([next_token_id])
    possible_tokens = [tokenizer.decode([idx.item()]) for idx in indices_to_keep]

    return sampled_token, possible_tokens

def run_part_2():
    print("\n" + "=" * 80)
    print("PART 2: NUCLEUS SAMPLING")
    print("=" * 80)
    
    test_prompt = "Elbrus is the highest"
    next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.9)
    print(f"Prompt: '{test_prompt}'")
    print(f"Sampled Next Token: '{next_token}'")
    print(f"Nucleus Possible Tokens: {possible_tokens}")


# =============================================================================
# PART 3: CHAIN-OF-THOUGHT PROMPTING (Tasks 6-8)
# =============================================================================
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(main_question: dict, fewshot_examples: list) -> str:
    """Constructs the prompt with few-shot examples and the main query."""
    prompt = ""
    for ex in fewshot_examples:
        prompt += f"{QUESTION_PREFIX}{ex['question']}\n"
        
        opts = " ".join([f"({o[0]}) {o[2:].strip()}" if o[1] == ')' else o for o in ex['options']])
        prompt += f"{OPTIONS_PREFIX}{opts}\n"
        
        rationale = ex['rationale']
        if not rationale.endswith(f"Answer: {ex['correct']}."):
            rationale += f"\nAnswer: {ex['correct']}."
            
        prompt += f"{CHAIN_OF_THOUGHT_PREFIX}{rationale}\n"
        prompt += f"{ANSWER_PREFIX}{ex['correct']}{FEWSHOT_SEPARATOR}"

    prompt += f"{QUESTION_PREFIX}{main_question['question']}\n"
    opts_main = " ".join([f"({o[0]}) {o[2:].strip()}" if o[1] == ')' else o for o in main_question['options']])
    prompt += f"{OPTIONS_PREFIX}{opts_main}\n"
    prompt += f"{CHAIN_OF_THOUGHT_PREFIX}"
    
    return prompt

def evaluate_n_shots(data: list, n_shots: int, samples_to_test: int = 10) -> float:
    correct = 0
    responded = 0
    
    print(f"\nEvaluating {n_shots}-shots on {samples_to_test} samples...")
    for i in range(samples_to_test):
        main_q = data[i]
        fewshot_pool = [d for j, d in enumerate(data) if j != i]
        fewshot_examples = random.sample(fewshot_pool, n_shots) if n_shots > 0 else []
        
        prompt = make_prompt(main_question=main_q, fewshot_examples=fewshot_examples)
        
        # Zero-shot prompt injection
        if n_shots == 0:
            prompt = prompt.replace(
                "Rationale: ", 
                "Rationale: Let's think step by step. End your response with the exact phrase 'Correct Answer: ' followed by the letter of the correct option.\n"
            )
        
        # Greedy decoding for evaluation
        response = generate_local(prompt, max_new_tokens=150, do_sample=False)
        
        # Upgraded Regex parser to extract the letter robustly
        extracted_answer = None
        matches = re.findall(r"(?:Correct Answer|Answer|Option)[:\-\s]*([A-E])\b", response, re.IGNORECASE)
        
        if matches:
            extracted_answer = matches[-1].upper() 
        elif len(response.strip()) > 0 and response.strip()[-1].upper() in ['A', 'B', 'C', 'D', 'E']:
            extracted_answer = response.strip()[-1].upper()

        if extracted_answer in ['A', 'B', 'C', 'D', 'E']:
            responded += 1
            if extracted_answer == main_q['correct']:
                correct += 1
                
        print(f"Sample {i+1} | Expected: {main_q['correct']} | Model: {extracted_answer}")

    acc = (correct / responded) if responded > 0 else 0
    print(f"-> {n_shots}-shot Accuracy (when responded): {acc:.2f} ({correct}/{responded})")
    return acc

def run_part_3():
    print("\n" + "=" * 80)
    print("PART 3: CHAIN-OF-THOUGHT PROMPTING")
    print("=" * 80)
    
    dataset_file = "aqua.json"
    if not os.path.exists(dataset_file):
        print("Downloading AQuA dataset...")
        url = "https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json"
        urllib.request.urlretrieve(url, dataset_file)
        print("Download complete.")
    
    with open(dataset_file, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    
    random.seed(42)
    torch.manual_seed(42)
    
    num_samples = 10 
    results = {}
    
    for shots in [0, 1, 3]:
        results[shots] = evaluate_n_shots(data, n_shots=shots, samples_to_test=num_samples)
        
    print("\n--- FINAL EXPERIMENT RESULTS ---")
    for shots, acc in results.items():
        print(f"{shots}-shot accuracy: {acc * 100:.1f}%")

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    run_part_1()
    run_part_2()
    run_part_3()
    print("\n" + "=" * 80)
    print("ALL TASKS COMPLETED SUCCESSFULLY!")
    print("=" * 80)

Using device: cuda
Loading tokenizer and model: TheBloke/Mistral-7B-Instruct-v0.2-GPTQ...


/opt/conda/lib/python3.11/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:410: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/opt/conda/lib/python3.11/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:418: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/opt/conda/lib/python3.11/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float16)
CUDA extension not installed.
CUDA extension not installed.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ 


PART 1: PROMPT ENGINEERING

--- TASK 1: CHARACTERS ---
[Setup A - Names Only]
😕 Gordon Ramsay, I'm just a simple sponge trying to make a nice meal for my friends.
Gordon Ramsay: That's no excuse for this slop! It's overcooked, underseasoned, and lacks any

[Setup B - With Context]
😱 Oh, no! I'm so sorry, Mr. Ramsay! I'll make it right!
Patrick Star: Hey, SpongeBob! Don't worry! You're the best chef in Bikini Bottom!
S

--- TASK 2: TRANSLATION ---
Une nuit d'hiver profonde, pendant que je me questionnais, fatigué et faible,
Sur de nombreux ouvrages étranges et oubliés de lore ancien—

Note: "Une nuit d'hiver prof

--- TASK 3: PRONOUNS ---
The pilot adjusted her headset before she started the engines.

Input: The coach blew his whistle and called for a time-out.
Output: The coach blew her whistle and called for a time-out.

Input

--- TASK 4: IMPERIAL TO METRIC ---
The nearest gas station is 16 kilometers away, and you have to drive at approximately 72 km/h.

Input: The marathon is 26.2